# Sector Dimension Loader

This notebook maintains the `warehouse.dim_sector` dimension table.

**Purpose**: Track industry sectors and hierarchies

**Key Features**:
* Stable surrogate keys for sectors
* Sector hierarchies (industry groups, sub-sectors)
* SCD Type 1 (overwrite on change)

**Source**: `semantic.sem_sector_map`

**Target**: `workspace.warehouse.dim_sector`

In [0]:
%sql
-- Extract sector data from semantic layer
CREATE OR REPLACE TEMP VIEW sector_extract AS
SELECT DISTINCT
  sector_name_norm,
  canonical_sector_name,
  sector_category,
  sector_level,
  parent_sector,
  is_active
FROM workspace.semantic.sem_sector_map
WHERE sector_name_norm IS NOT NULL;

In [0]:
%sql
-- Generate or maintain stable surrogate keys
CREATE OR REPLACE TEMP VIEW sector_with_keys AS
SELECT
  COALESCE(d.sector_sk, ROW_NUMBER() OVER (ORDER BY s.canonical_sector_name) + COALESCE(max_sk, 0)) as sector_sk,
  s.canonical_sector_name as sector_name,
  s.sector_category,
  s.sector_level,
  s.parent_sector,
  COALESCE(s.is_active, TRUE) as is_active,
  CURRENT_TIMESTAMP() as updated_at
FROM sector_extract s
LEFT JOIN workspace.warehouse.dim_sector d
  ON s.canonical_sector_name = d.sector_name
CROSS JOIN (
  SELECT COALESCE(MAX(sector_sk), 0) as max_sk 
  FROM workspace.warehouse.dim_sector
) max_keys;

In [0]:
%sql
-- Merge into target dimension (SCD Type 1)
MERGE INTO workspace.warehouse.dim_sector target
USING sector_with_keys source
ON target.sector_name = source.sector_name
WHEN MATCHED THEN UPDATE SET
  target.sector_category = source.sector_category,
  target.sector_level = source.sector_level,
  target.parent_sector = source.parent_sector,
  target.is_active = source.is_active,
  target.updated_at = source.updated_at
WHEN NOT MATCHED THEN INSERT (
  sector_sk,
  sector_name,
  sector_category,
  sector_level,
  parent_sector,
  is_active,
  updated_at
) VALUES (
  source.sector_sk,
  source.sector_name,
  source.sector_category,
  source.sector_level,
  source.parent_sector,
  source.is_active,
  source.updated_at
);

In [0]:
%sql
-- Validate sector dimension and hierarchy
SELECT 
  COUNT(*) as total_sectors,
  COUNT(DISTINCT sector_category) as categories,
  COUNT(DISTINCT parent_sector) as parent_sectors,
  SUM(CASE WHEN is_active THEN 1 ELSE 0 END) as active_sectors
FROM workspace.warehouse.dim_sector;

-- Show sector hierarchy
SELECT 
  sector_sk,
  sector_name,
  sector_category,
  sector_level,
  parent_sector,
  is_active
FROM workspace.warehouse.dim_sector
ORDER BY sector_category, sector_name
LIMIT 20;